In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## Statistics by City

In [ ]:
import pandas as pd
import glob
import os
import matplotlib.pyplot as plt

folder = "/content/drive/MyDrive/Summary Statistics"

files = glob.glob(os.path.join(folder, "*.csv"))
print(f"Found {len(files)} CSV files")

dfs = []
for f in files:
    df = pd.read_csv(f)
    dfs.append(df)

combined = pd.concat(dfs, ignore_index=True)

# Aggregate by city
summary = (
    combined.groupby("Region")
    .agg(
        mean_morans_i=("Mean Moran's I", "mean"),
        median_morans_i=("Mean Moran's I", "median"),
        mean_segments=("n (segments)", "mean"),
        snapshots=("Date", "count")
    )
    .reset_index()
)

# Clean labels
summary["Region"] = summary["Region"].str.replace("_", " ", regex=False)

summary = summary.rename(columns={
    "Region": "City",
    "mean_morans_i": "Mean Moran’s I",
    "median_morans_i": "Median Moran’s I",
    "mean_segments": "Mean # Segments",
    "snapshots": "# Snapshots"
})

# Format values
summary["Mean Moran’s I"] = summary["Mean Moran’s I"].round(3)
summary["Median Moran’s I"] = summary["Median Moran’s I"].round(3)
summary["Mean # Segments"] = summary["Mean # Segments"].round(0).astype(int)

summary = summary.sort_values("City").reset_index(drop=True)

print(summary)

# Save output
out_csv = "/content/drive/MyDrive/morans_city_summary.csv"
summary.to_csv(out_csv, index=False)
print(f"Saved to: {out_csv}")

## Statistics by AM and PM


In [ ]:
# Aggregate by city and travel period (AM vs PM)

combined["Date"] = pd.to_datetime(combined["Date"])
combined["Hour"] = combined["Date"].dt.hour

def classify_period(hour):
    if hour in [7, 8, 9]:
        return "AM"
    elif hour in [16, 17, 18]:
        return "PM"
    else:
        return None

combined["Travel Period"] = combined["Hour"].apply(classify_period)

travel_summary = (
    combined.groupby(["Region", "Travel Period"])
    .agg(
        mean_morans_i=("Mean Moran's I", "mean"),
        median_morans_i=("Mean Moran's I", "median"),
        mean_segments=("n (segments)", "mean"),
        snapshots=("Date", "count")
    )
    .reset_index()
)

travel_summary["Region"] = travel_summary["Region"].str.replace("_", " ", regex=False)

travel_summary = travel_summary.rename(columns={
    "Region": "City",
    "mean_morans_i": "Mean Moran’s I",
    "median_morans_i": "Median Moran’s I",
    "mean_segments": "Mean # Segments",
    "snapshots": "# Snapshots"
})

travel_summary["Mean Moran’s I"] = travel_summary["Mean Moran’s I"].round(3)
travel_summary["Median Moran’s I"] = travel_summary["Median Moran’s I"].round(3)
travel_summary["Mean # Segments"] = travel_summary["Mean # Segments"].round(0).astype(int)

travel_summary = travel_summary.sort_values(["City", "Travel Period"]).reset_index(drop=True)

travel_summary

## Box Plot by AM and PM

In [ ]:
folder = "/content/drive/MyDrive/Summary Statistics"

files = glob.glob(os.path.join(folder, "*.csv"))
dfs = [pd.read_csv(f) for f in files]
combined = pd.concat(dfs, ignore_index=True)

combined["Region"] = combined["Region"].str.replace("_", " ", regex=False)
combined["Date"] = pd.to_datetime(combined["Date"])
combined["Hour"] = combined["Date"].dt.hour
combined["Travel Period"] = combined["Hour"].apply(lambda h: "AM" if h in [7, 8, 9] else "PM")

city_order = [
    "Fresno",
    "Los Angeles",
    "Sacramento",
    "San Diego",
    "San Francisco",
    "San Jose"
]

fig, axes = plt.subplots(1, 2, figsize=(14, 6), sharey=True)

for ax, period in zip(axes, ["AM", "PM"]):
    subset = combined[combined["Travel Period"] == period].copy()
    subset["Region"] = pd.Categorical(subset["Region"], categories=city_order, ordered=True)
    subset.boxplot(column="Mean Moran's I", by="Region", ax=ax)
    ax.set_title(period)
    ax.set_xlabel("City")
    ax.set_ylabel("Global Moran’s I")
    ax.tick_params(axis="x", rotation=30)
    ax.grid(True)

plt.suptitle("Distribution of Global Moran’s I by City and Travel Period\n(March 17–29, 2026)")
plt.tight_layout()
plt.show()